In [1]:
import csv
from pathlib import Path

In [2]:
actual_path = Path("raw-zip-actual")
actual_files = sorted(actual_path.glob("*.zip"), reverse=True)
print(len(actual_files))
actual_files[:5]

230


[PosixPath('raw-zip-actual/20260201RTLineOutages_csv.zip'),
 PosixPath('raw-zip-actual/20260101RTLineOutages_csv.zip'),
 PosixPath('raw-zip-actual/20251201RTLineOutages_csv.zip'),
 PosixPath('raw-zip-actual/20251101RTLineOutages_csv.zip'),
 PosixPath('raw-zip-actual/20251001RTLineOutages_csv.zip')]

In [3]:
scheduled_path = Path("raw-zip-scheduled")
scheduled_files = sorted(scheduled_path.glob("*.zip"), reverse=True)
print(len(scheduled_files))
scheduled_files[:5]

218


[PosixPath('raw-zip-scheduled/20260201SCLineOutages_csv.zip'),
 PosixPath('raw-zip-scheduled/20260101SCLineOutages_csv.zip'),
 PosixPath('raw-zip-scheduled/20251201SCLineOutages_csv.zip'),
 PosixPath('raw-zip-scheduled/20251101SCLineOutages_csv.zip'),
 PosixPath('raw-zip-scheduled/20251001SCLineOutages_csv.zip')]

In [4]:
import re
from collections import defaultdict, namedtuple

ActualOutage = namedtuple(
    "ActualOutage",
    [
        "timestamp",
        "ptid",
        "equipment_name",
        "outage_datetime",
        "bus1",
        "bus2",
        "voltage",
    ],
)

ScheduledOutage = namedtuple(
    "ScheduledOutage",
    [
        "timestamp",
        "ptid",
        "equipment_name",
        "scheduled_out_datetime",
        "scheduled_in_datetime",
        "bus1",
        "bus2",
        "voltage",
    ],
)


equipment_name_pattern = r"^([A-Za-z0-9._]{8})-([A-Za-z0-9._]{8})_(\d{2,3})_(.+)$"

In [5]:
zip_path = actual_files[0]

In [6]:
import io
from datetime import datetime
from zipfile import ZipFile


def parse_actual_outage(row):
    equipment_name = row["Equipment Name"]
    m = re.match(equipment_name_pattern, equipment_name)
    if m is None:
        return None
    return ActualOutage(
        timestamp=datetime.strptime(row["Timestamp"], "%m/%d/%Y %H:%M:%S"),
        ptid=int(row["PTID"]),
        equipment_name=row["Equipment Name"],
        outage_datetime=datetime.strptime(row["Outage Date/Time"], "%m/%d/%Y %H:%M:%S"),
        bus1=m.group(1),
        bus2=m.group(2),
        voltage=int(m.group(3)),
    )


def parse_scheduled_outage(row):
    equipment_name = row["Equipment Name"]
    m = re.match(equipment_name_pattern, equipment_name)
    if m is None:
        return None
    return ScheduledOutage(
        timestamp=datetime.strptime(row["Timestamp"], "%m/%d/%Y %H:%M:%S"),
        ptid=int(row["PTID"]),
        equipment_name=row["Equipment Name"],
        scheduled_out_datetime=datetime.strptime(
            row["Scheduled Out Date/Time"],
            "%m/%d/%Y %H:%M:%S",
        ),
        scheduled_in_datetime=datetime.strptime(
            row["Scheduled In Date/Time"],
            "%m/%d/%Y %H:%M:%S",
        ),
        bus1=m.group(1),
        bus2=m.group(2),
        voltage=int(m.group(3)),
    )


def list_csvs(zip_path):
    with ZipFile(zip_path) as z:
        return [i for i in sorted(z.namelist()) if i.endswith(".csv")]


def read_csv_from_zip(zip_path, csv_name, parse_row):
    with ZipFile(zip_path) as z:
        with z.open(csv_name) as f:
            data = f.read()
        csv_reader = csv.DictReader(io.StringIO(data.decode("utf-8")))
        # Only keep rows with line outages
        data = [parse_row(row) for row in csv_reader]
        data = [row for row in data if row is not None]
    return data


# Example using an existing variable in the notebook:
zip_path = actual_files[0]
print(zip_path)
print("members:", list_csvs(zip_path))

# Read one file from the zip (text)
member = list_csvs(zip_path)[0]
data = read_csv_from_zip(zip_path, member, parse_actual_outage)

raw-zip-actual/20260201RTLineOutages_csv.zip
members: ['20260201RTLineOutages.csv', '20260202RTLineOutages.csv', '20260203RTLineOutages.csv', '20260204RTLineOutages.csv', '20260205RTLineOutages.csv', '20260206RTLineOutages.csv', '20260207RTLineOutages.csv', '20260208RTLineOutages.csv', '20260209RTLineOutages.csv', '20260210RTLineOutages.csv', '20260211RTLineOutages.csv', '20260212RTLineOutages.csv', '20260213RTLineOutages.csv', '20260214RTLineOutages.csv', '20260215RTLineOutages.csv', '20260216RTLineOutages.csv', '20260217RTLineOutages.csv', '20260218RTLineOutages.csv', '20260219RTLineOutages.csv', '20260220RTLineOutages.csv', '20260221RTLineOutages.csv', '20260222RTLineOutages.csv', '20260223RTLineOutages.csv', '20260224RTLineOutages.csv']


# Compress redundant rows

In [7]:
by_ptid = defaultdict(list)

for outage in data:
    by_ptid[outage.ptid].append(outage)

by_ptid.keys()

dict_keys([25013, 25015, 25020, 25033, 25038, 25048, 25129, 25134, 25243, 25263, 25264, 25271, 25313, 25349, 25426, 25506, 25524, 25559, 25560, 25561, 25562, 25565, 25566, 25769, 25770, 25830, 25875, 25876, 26054, 26148, 26187, 26256, 26321, 26332, 70027, 325252, 325649, 326129, 326145, 326191, 326194, 326407, 326488, 326577, 326633, 326639, 326710, 326738, 326866, 327143, 625081, 625165, 625177, 625179, 625182, 625186, 625194, 625195, 625205, 625226, 625229, 101015020, 101018431, 101018432, 101018433, 101018434, 25059, 26192, 327195])

In [8]:
# separate ptid by outage date
by_ptid_and_outage_date = defaultdict(lambda: defaultdict(list))
for ptid, rows in by_ptid.items():
    for row in rows:
        by_ptid_and_outage_date[ptid][row.outage_datetime].append(row)
    # check partition length be at least two, for beginning and ending
    for partition_key in by_ptid_and_outage_date[ptid].keys():
        partition_list = by_ptid_and_outage_date[ptid][row.outage_datetime]
        if len(partition_list) == 1:
            partition_list.append(partition_list[0])

In [12]:
# compression by skipping redundancies data
Interval = namedtuple("Interval", ["start", "end"])

summary = {}
for ptid, date_log_dd in by_ptid_and_outage_date.items():
    partitions = {}
    for date, log_list in date_log_dd.items():
        interval = Interval(log_list[0], log_list[-1])
        partitions[date] = interval
    summary[ptid] = partitions

In [14]:
for k, v in summary.items():
    if len(v) > 1:
        print(k)

26192


In [15]:
summary[26192]

{datetime.datetime(2026, 2, 1, 7, 56): Interval(start=ActualOutage(timestamp=datetime.datetime(2026, 2, 1, 8, 7), ptid=26192, equipment_name='TULLERHL-CLRKSCRN_115_1-716', outage_datetime=datetime.datetime(2026, 2, 1, 7, 56), bus1='TULLERHL', bus2='CLRKSCRN', voltage=115), end=ActualOutage(timestamp=datetime.datetime(2026, 2, 1, 11, 52), ptid=26192, equipment_name='TULLERHL-CLRKSCRN_115_1-716', outage_datetime=datetime.datetime(2026, 2, 1, 7, 56), bus1='TULLERHL', bus2='CLRKSCRN', voltage=115)),
 datetime.datetime(2026, 2, 1, 8, 3): Interval(start=ActualOutage(timestamp=datetime.datetime(2026, 2, 1, 8, 7), ptid=26192, equipment_name='TULLERHL-CLRKSCRN_115_1-716', outage_datetime=datetime.datetime(2026, 2, 1, 8, 3), bus1='TULLERHL', bus2='CLRKSCRN', voltage=115), end=ActualOutage(timestamp=datetime.datetime(2026, 2, 1, 11, 52), ptid=26192, equipment_name='TULLERHL-CLRKSCRN_115_1-716', outage_datetime=datetime.datetime(2026, 2, 1, 8, 3), bus1='TULLERHL', bus2='CLRKSCRN', voltage=115))}

In [10]:
# from tqdm import tqdm

# actual_outages = []
# for zip_path in tqdm(actual_files):
#     for member in list_csvs(zip_path):
#         rows = read_csv_from_zip(zip_path, member, parse_actual_outage)
#         actual_outages.extend(rows)